# 01 — Load datasets and make diagnostic histograms

**Stage 0/1 validation notebook** for the BNV searches
$B^0 \to \Lambda^0 \Lambda^0$ and $B^+ \to \Lambda_c^+ \Lambda^0$.

This notebook:
1. loads the MC (SP) and **blinded** collision-data parquet files for one channel,
2. makes an inventory of events per SP mode and candidate multiplicities,
3. computes the MC luminosity-scaling weights,
4. fills and plots diagnostic histograms of the variables we will work with,
5. shows $M_{ES}$ vs. $\Delta E$ for MC and **verifies the blinding** of the collision file.

**Blinding note:** only the `_BLINDED` collision files are ever opened here
(enforced in `datasets.load_datasets`). The signal-region count in data must be zero.

In [ ]:
import sys
sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

import numpy as np
import awkward as ak
import pandas as pd
import matplotlib.pylab as plt

from channel_config import (get_channel_config, SP_MODE_INFO,
                            BACKGROUND_SP_MODES, SIGNAL_SP_MODE, DATA_SP_MODE)
import datasets
import cutflow
import plotting

In [ ]:
# Select the channel here: 'Lam0Lam0' or 'Lam0LamC'
CHANNEL = 'Lam0Lam0'

config = get_channel_config(CHANNEL)
print(f"Channel: {config['name']}   {config['decay_label']}")
print()
print("Region definitions:")
for k, v in config['region_definitions'].items():
    if k != 'inference':
        print(f"  {k:22s} {v}")

## Load the datasets

In [ ]:
data_sp, data_collision = datasets.load_datasets(CHANNEL)

# Channel-dependent derived fields (LambdaC flight significance,
# Lambda0-from-B vs Lambda0-from-LambdaC flight significance, ...)
datasets.add_derived_fields(data_sp, config)
datasets.add_derived_fields(data_collision, config)

print(f"MC (SP) events:    {len(data_sp)}")
print(f"Collision events:  {len(data_collision)}  (BLINDED file)")
print(f"# of fields:       {len(data_sp.fields)}")

## Event inventory

Raw (unweighted) numbers of events per SP mode, and the candidate
multiplicities before any cuts.

In [ ]:
df_sp_counts = datasets.event_counts_by_spmode(data_sp)
df_col_counts = datasets.event_counts_by_spmode(data_collision)

df_sp_counts['label'] = [SP_MODE_INFO.get(s, {}).get('label', '?') for s in df_sp_counts['spmode']]

print("MC (SP modes):")
display(df_sp_counts)
print("Collision data:")
display(df_col_counts)

In [ ]:
# Candidate multiplicities (no cuts), MC vs data
multiplicity_vars = ['nB'] + [f"n{name}" for name in config['composites'].keys()]

fig, axes = plt.subplots(1, len(multiplicity_vars), figsize=(5 * len(multiplicity_vars), 3.5))

for ax, var in zip(np.atleast_1d(axes), multiplicity_vars):
    for label, arr in [('MC (all SP)', data_sp), ('Data', data_collision)]:
        x = np.array(arr[var].to_list())
        vals, counts = np.unique(x, return_counts=True)
        ax.plot(vals, counts / len(x), 'o-', label=label)
    ax.set_xlabel(var)
    ax.set_ylabel('fraction of events')
    ax.set_yscale('log')
    ax.legend()
plt.tight_layout()

## MC scaling weights

Background MC is weighted to the integrated luminosity of the data
(cross section $\times$ luminosity / number generated).
Signal MC is filled with weight 1; its normalization on the plots is
**arbitrary** -- `plot_stacked` rescales it per panel so its peak matches
the summed-background peak (labeled "arb. norm.").

In [ ]:
bkg_spmodes_present = [s for s in BACKGROUND_SP_MODES if s in df_sp_counts['spmode'].values]

weights = datasets.get_scaling_weights(bkg_spmodes_present, verbose=False)

# Signal MC: fill with weight 1 -- visibility scaling happens at plot time
weights[SIGNAL_SP_MODE] = 1.0

# Collision data is unweighted (whatever spmode label it carries)
for s in df_col_counts['spmode'].values:
    weights[str(s)] = 1.0

pd.DataFrame({'spmode': list(weights.keys()),
              'weight': list(weights.values())})

## Diagnostic cuts and cutflow

Minimal cuts for the first look:

| cut | definition |
|-----|------------|
| 0 | none |
| 1 | single-candidate requirement (`nB == 1`, expected number of $\Lambda$'s) |
| 2 | 1 + B candidate in the $M_{ES}/\Delta E$ fitting region |

In [ ]:
dcuts_sp = cutflow.build_diagnostic_cuts(data_sp, config)
dcuts_col = cutflow.build_diagnostic_cuts(data_collision, config)

df_cutflow_sp = cutflow.get_numbers_for_cut_flow(data_sp, dcuts_sp, tag=CHANNEL)
df_cutflow_col = cutflow.get_numbers_for_cut_flow(data_collision, dcuts_col, tag=CHANNEL)

print("MC cutflow (raw events):")
display(df_cutflow_sp.pivot_table(index=['cut', 'name'], columns='spmode', values='nevents'))
print("Collision-data cutflow:")
display(df_cutflow_col.pivot_table(index=['cut', 'name'], columns='spmode', values='nevents'))

## Fill the diagnostic histograms

In [ ]:
all_hists = plotting.create_empty_histograms(config['hist_defs'])

df_fill_sp = plotting.fill_histograms(data_sp, all_hists, dcuts_sp, weights=weights)
df_fill_col = plotting.fill_histograms(data_collision, all_hists, dcuts_col, weights=weights)

print(f"Filled {df_fill_sp['var'].nunique()} variables for MC, "
      f"{df_fill_col['var'].nunique()} for data")

### Candidate multiplicities before any cuts

Number of B and $\Lambda$ candidates per event (cut 0 = no cuts), stacked
by SP mode with data overlaid. This is what the single-candidate
requirement acts on.

In [ ]:
multiplicity_vars = ['nB'] + [f"n{name}" for name in config['composites'].keys()]

plotting.plot_stacked_grid(all_hists, vars=multiplicity_vars, cut='0', ncols=3,
                           save=True, config=config, extra_tag='_multiplicity',
                           logy=True)

### Candidate kinematics (after single-candidate cut)

Stacked, luminosity-weighted background MC; signal MC (arbitrary
normalization) as the solid line; blinded collision data as points.

In [ ]:
kinematic_vars = ['BpostFitMes', 'BpostFitDeltaE', 'BpostFitMass',
                  'Lambda0_unc_Mass', 'Lambda0FlightLen', 'Lambda0postFitFlight',
                  'Lambda0postFitFlightSignificance', 'Lambda0p3CM']
if CHANNEL == 'Lam0LamC':
    kinematic_vars += ['LambdaC_unc_Mass', 'LambdaCFlightLen']

plotting.plot_stacked_grid(all_hists, vars=kinematic_vars, cut='1', ncols=3,
                           save=True, config=config, extra_tag='_kinematics')

### Event-shape / continuum-suppression variables

These are the candidate MLP inputs (Stage 5). Look for gross data/MC
disagreements and outliers now, while it is cheap to fix.

In [ ]:
eventshape_vars = ['R2', 'R2All', 'thrustMag', 'thrustMagAll',
                   'thrustCosTh', 'thrustCosThAll', 'sphericityAll',
                   'BSphr', 'BCosSphr', 'BThrust', 'BCosThrust',
                   'BCosThetaS', 'BCosThetaT', 'BLegendreP2',
                   'BR2ROE', 'BSphrROE', 'BThrustROE',
                   'nTRK', 'nGoodTrkLoose']

plotting.plot_stacked_grid(all_hists, vars=eventshape_vars, cut='1', ncols=4,
                           save=True, config=config, extra_tag='_eventshape')

## $\Lambda_c^+$ decay modes (`Lam0LamC` only)

Signal MC was generated with **equal amounts** of four $\Lambda_c^+$ decay
modes, and candidates are reconstructed in any of them. We classify each
candidate from `LambdaCnDaus` and `LambdaCd1Lund`:

| mode | decay | `nDaus` | \|`d1Lund`\| |
|------|-------|---------|--------------|
| 1 | $p K^- \pi^+$ | 3 | 2212 |
| 2 | $p K_S^0$ | 2 | 2212 |
| 3 | $p K_S^0 \pi^+\pi^-$ | 4 | 2212 |
| 4 | $\Lambda^0 \pi^+\pi^+\pi^-$ | 4 | 3122 |

The table below compares the mode fractions in signal MC before any cuts
and after the single-candidate requirement. **Expect mode 4 to be
suppressed by the `nLambda0 == 1` requirement** (its $\Lambda^0$ daughter
enters the same collection as the $B$'s direct $\Lambda^0$) -- this is a
design item for the Stage 1 discussion.

*(These cells do nothing when `CHANNEL == 'Lam0Lam0'`.)*

In [ ]:
if CHANNEL == 'Lam0LamC':
    lamc_mode = cutflow.get_lambdac_decay_mode(data_sp)
    mask_sig = data_sp['spmode'] == SIGNAL_SP_MODE

    def mode_fractions(mode_flat):
        vals, counts = np.unique(ak.to_numpy(mode_flat), return_counts=True)
        return {int(v): c / counts.sum() for v, c in zip(vals, counts)}

    # All LambdaC candidates in signal MC, before any cuts
    frac_nocut = mode_fractions(ak.flatten(lamc_mode[mask_sig]))

    # After the single-candidate requirement
    frac_cut1 = mode_fractions(ak.flatten(lamc_mode[dcuts_sp[1]['event'] & mask_sig]))

    df_modes = pd.DataFrame({
        'mode': list(config['lambdac_modes'].keys()),
        'decay': list(config['lambdac_modes'].values()),
        'frac (no cut)': [frac_nocut.get(m, 0.0) for m in config['lambdac_modes']],
        'frac (single cand.)': [frac_cut1.get(m, 0.0) for m in config['lambdac_modes']],
    })
    display(df_modes)

In [ ]:
if CHANNEL == 'Lam0LamC':
    mask_sig = data_sp['spmode'] == SIGNAL_SP_MODE
    mode_perB = cutflow.get_lambdac_decay_mode_per_B(data_sp, config)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

    # LambdaC mass per LambdaC candidate
    plotting.plot_split_by_mode(ak.flatten(data_sp['LambdaC_unc_Mass'][mask_sig]),
                                ak.flatten(lamc_mode[mask_sig]),
                                config['hist_defs']['LambdaC_unc_Mass'],
                                config['lambdac_modes'], ax=axes[0],
                                title='Signal MC (no cuts)')

    # mES and DeltaE per B candidate, tagged by its LambdaC daughter's mode
    for ax, var in zip(axes[1:], ['BpostFitMes', 'BpostFitDeltaE']):
        plotting.plot_split_by_mode(ak.flatten(data_sp[var][mask_sig]),
                                    ak.flatten(mode_perB[mask_sig]),
                                    config['hist_defs'][var],
                                    config['lambdac_modes'], ax=ax,
                                    title='Signal MC (no cuts)')

    plt.tight_layout()
    plt.savefig(f"{plotting.plot_dir(config)}/lambdac_mode_split.png", dpi=150)

### Flight significances and $K_S^0$ (stacked by SP mode, no cuts)

- $\Lambda^0$ flight significance, separately for $\Lambda^0$'s that are
  direct $B$ daughters vs. those used inside a $\Lambda_c^+$ (mode 4),
- $\Lambda_c^+$ flight significance (`FlightLen/FlightErr` -- no post-fit
  branch exists for the $\Lambda_c^+$),
- $K_S^0$ **pre-fit** mass and flight significance (modes 2 and 3).
  (`K_SMass` is the post-fit, mass-constrained value -- just a spike at
  the PDG mass, so we plot `K_SpreFitMass`.)

In [ ]:
if CHANNEL == 'Lam0LamC':
    lamc_extra_vars = ['Lambda0FLSig_fromB', 'Lambda0FLSig_fromLambdaC',
                       'LambdaCFlightSignificance',
                       'K_SpreFitMass', 'K_SpostFitFlightSignificance']

    # cut '0' (no cuts): the single-candidate requirement suppresses mode 4,
    # which would empty the Lambda0-from-LambdaC panel
    plotting.plot_stacked_grid(all_hists, vars=lamc_extra_vars, cut='0', ncols=3,
                               save=True, config=config, extra_tag='_lamc_specific')

## $M_{ES}$ vs. $\Delta E$ (MC)

In [ ]:
mask_sig = data_sp['spmode'] == SIGNAL_SP_MODE

plt.figure(figsize=(13, 4.5))

plt.subplot(1, 2, 1)
counts_sig = plotting.plot_mes_vs_DeltaE(
    ak.flatten(data_sp['BpostFitMes'][mask_sig]),
    ak.flatten(data_sp['BpostFitDeltaE'][mask_sig]),
    config, draw_signal_region=True, draw_sidebands=True, zoom=True,
    title='Signal MC', verbose=True)

plt.subplot(1, 2, 2)
counts_bkg = plotting.plot_mes_vs_DeltaE(
    ak.flatten(data_sp['BpostFitMes'][~mask_sig]),
    ak.flatten(data_sp['BpostFitDeltaE'][~mask_sig]),
    config, draw_signal_region=True, draw_sidebands=True, zoom=True,
    title='Background MC (all modes, unweighted)', verbose=True)

plt.tight_layout()
plt.savefig(f"{plotting.plot_dir(config)}/mes_vs_de_MC.png", dpi=150)

## Blinding verification (collision data)

The collision file was blinded upstream; here we **verify** that
(a) the signal-region count is exactly zero and (b) the empty box in the
2D plot fully covers the signal window assumed in `channel_config.py`.
If either fails, STOP and resolve the window definitions before doing
anything else with the data.

In [ ]:
plt.figure(figsize=(6.5, 4.5))
counts_data = plotting.plot_mes_vs_DeltaE(
    ak.flatten(data_collision['BpostFitMes']),
    ak.flatten(data_collision['BpostFitDeltaE']),
    config, draw_signal_region=True, draw_sidebands=True, zoom=True,
    title='Collision data (BLINDED)', verbose=True)
plt.savefig(f"{plotting.plot_dir(config)}/mes_vs_de_data_blinded.png", dpi=150)

if counts_data['nsig'] == 0:
    print("\nBlinding check PASSED: 0 data candidates in the assumed signal region.")
else:
    print("\n*** WARNING: BLINDING CHECK FAILED ***")
    print(f"{counts_data['nsig']} data candidates fall inside the signal window")
    print("assumed in channel_config.py. Either the file was blinded with a")
    print("narrower window than assumed, or something is wrong.")
    print("STOP and resolve the window definitions before proceeding.")

## Observations / checkpoint summary

*(fill in after running -- items for the Stage 0/1 review)*

- [ ] Event counts per SP mode as expected?
- [ ] Candidate multiplicities: how often is the single-candidate
      requirement satisfied? (Especially for `Lam0LamC`, where up to
      60 B candidates/event were seen.)
- [ ] `Lam0LamC`: are the four $\Lambda_c^+$ decay modes present in the
      expected (equal) proportions in signal MC -- and how does the
      single-candidate requirement distort them? (Watch mode 4,
      $\Lambda^0\pi^+\pi^+\pi^-$: the `nLambda0 == 1` requirement is
      expected to suppress it.)
- [ ] Any variables with outliers / pathological distributions?
- [ ] Data/MC agreement reasonable in the event-shape variables?
      (A known overall scale offset of order 2x was seen in previous work.)
- [ ] Blinding verification passed, and blinded box covers the assumed
      signal window?
- [ ] Signal MC peaks where expected in $M_{ES}$, $\Delta E$,
      $\Lambda^0$ (and $\Lambda_c^+$, $K_S^0$) mass?

**Next steps:** review at stage checkpoint, then Stage 1 (preselection /
single-candidate treatment) and Stage 2 ($\Lambda$ purity studies).